# Day 17 - TTS Voice & Multimodal Search

## RetailMind AI Increment

Day 17 completes the generation track by adding two customer-facing capabilities:

1. **Text-to-speech (TTS):** Day-14 support answers can be rendered as voice replies.
2. **CLIP multimodal search:** customers can search the catalog using either a natural-language description or a reference image.

### Business objective
Voice support improves accessibility and hands-free interaction, while CLIP allows customers to search by meaning and appearance rather than relying only on exact catalog keywords.

## Dataset

The image side uses **Fashion Product Images Small** or approved catalog photographs. The text side uses each product's name and description.

Expected JSON:

```json
{
  "products": [
    {
      "product_id": "P1001",
      "product_name": "Black Running Shoe",
      "description": "Lightweight mesh running shoe with white sole",
      "image_path": "data/catalog/P1001.jpg"
    }
  ]
}
```

CLIP places both text and images into a shared embedding space, allowing cross-modal similarity.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
import day17_tts_multimodal_search as day17

print("CLIP model:", day17.CLIP_MODEL_ID)
print("TTS model:", day17.TTS_MODEL_ID)
print("Default top-k:", day17.TOP_K)

## 1. Load and audit the catalog

Every product requires an ID, name, description, and existing image path. Keeping a stable product ID is essential because vector search results must map back to real catalog records.

In [ ]:
# Run when catalog files are present.
# products = day17.load_catalog(repo_root / day17.CATALOG_PATH)
# print("Products:", len(products))
# print(products[0])

## 2. CLIP shared embedding space

CLIP jointly learns visual and textual concepts. A text prompt such as “lightweight black running shoes” and an image of a visually related shoe can therefore be compared to the same product index.

The RetailMind index retains three vectors:

- image embedding;
- catalog-text embedding;
- normalized average of image + text, used as the default product vector.

In [ ]:
# Requires pretrained CLIP weights/cache.
# clip_model, clip_processor = day17.load_clip(local_files_only=False)
# image_emb, text_emb, combined_emb = day17.build_catalog_index(products, clip_model, clip_processor)
# print("Image embeddings:", image_emb.shape)
# print("Text embeddings:", text_emb.shape)
# print("Combined embeddings:", combined_emb.shape)

## 3. Search by text

A customer's natural-language query is encoded with CLIP's text encoder and ranked against the product vectors using cosine similarity. Because embeddings are normalized, the dot product equals cosine similarity.

In [ ]:
# results = day17.search_by_text(
#     "lightweight black shoes for running",
#     clip_model, clip_processor, products, combined_emb.cpu().numpy(), k=5
# )
# for item in results:
#     print(item["rank"], round(item["score"], 3), item["product_name"])

## 4. Search by image

A reference image is passed through CLIP's vision encoder and ranked against the same product index. This lets a customer upload a visually similar product and retrieve related catalog items without knowing the exact product vocabulary.

In [ ]:
# results = day17.search_by_image(
#     "data/query/reference_product.jpg",
#     clip_model, clip_processor, products, combined_emb.cpu().numpy(), k=5
# )
# for item in results:
#     print(item["rank"], round(item["score"], 3), item["product_name"])

## 5. Evaluate multimodal retrieval

Do not judge search quality from one attractive example. Build a small query set with a known relevant product and report **Recall@K** or **Hit@K**.

Example:

```text
Query: "black running shoe"      Relevant: P1001
Query image: shoe_reference.jpg  Relevant: P1001
```

For 20 labeled queries, Recall@5 is the proportion where the correct product appears in the first five results.

In [ ]:
def recall_at_k(result_lists, relevant_ids, k=5):
    hits = 0
    for results, relevant in zip(result_lists, relevant_ids):
        retrieved = [item["product_id"] for item in results[:k]]
        hits += relevant in retrieved
    return hits / len(relevant_ids) if relevant_ids else 0.0

# print("Recall@5:", recall_at_k(search_results, relevant_product_ids, 5))

## 6. Add TTS voice replies

The support assistant from Day 14 already produces grounded text. Day 17 converts that text to audio rather than asking a new generative model to invent another answer. This preserves the Day-14 grounding decision while adding an accessible output channel.

In [ ]:
# Requires the Hugging Face TTS model/cache.
# tts = day17.load_tts(local_files_only=False)
# voice_path = day17.speak_reply(
#     "Your return request is eligible under the supplied policy. I can guide you through the next step.",
#     tts,
#     "return_support_reply.wav",
# )
# print("Saved:", voice_path)

## 7. Voice-quality checks

Before exposing voice replies, review:

- intelligibility and clipping;
- pronunciation of product/model names;
- numbers, dates, currency and abbreviations;
- response length;
- whether spoken content exactly reflects the grounded text answer;
- accessibility controls such as replay, pause and visible text transcript.

TTS should be an additional interface, not a replacement for the visible answer.

## 8. Wire voice and multimodal search into RetailMind

The Day-17 bridge can accept a customer question and an already-grounded Day-14 support response, search the catalog for related products, and optionally synthesize the support response to WAV.

```text
Customer question
      |      | -> Day 14 grounded support answer -> Day 17 TTS -> voice reply
      |
      -> CLIP text/image embedding -> catalog similarity -> related products
```

This completes the generation track: **text, images, variations/video frames, voice, and multimodal retrieval**.

## 9. Limitations and production controls

- CLIP similarity is relevance, not proof that two products are interchangeable.
- Ambiguous queries can return visually similar but functionally different items.
- Catalog vectors must be regenerated when product images/descriptions materially change.
- TTS pronunciation should be tested for brand and SKU names.
- Pretrained model licenses and organizational deployment policies must be checked before production use.
- Search should retain standard keyword/filter controls for price, inventory, size and other exact attributes that embeddings may not enforce reliably.

### Day 18 handoff
Phase 4 can now add a vector-database knowledge base for grounded retrieval, reusing the same general embedding-and-search pattern for enterprise knowledge rather than catalog products.